In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_colwidth', None)

In [2]:
icd9 = pd.read_json('../data/icd9-cm-2015-code.json').T
icd9 = icd9.reset_index(names="Code")
icd9

/tmp/ipykernel_2065915/641046097.py:1: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  icd9 = pd.read_json('../data/icd9-cm-2015-code.json').T
/tmp/ipykernel_2065915/641046097.py:1: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  icd9 = pd.read_json('../data/icd9-cm-2015-code.json').T
/tmp/ipykernel_2065915/641046097.py:1: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matchin

,Code,code,title,exclude,sex,synonym,include,codeAlso,note,fourth-digit
0,00,00,"Procedures and interventions , Not Elsewhere Classified",NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,000,00.0,Therapeutic ultrasound,"[diagnostic ultrasound (non-invasive) (88.71-88.79), intracardiac echocardiography [ICE] (heart chamber(s)) (37.28), intravascular imaging (adjunctive) (00.21-00.29)]",NaN,NaN,NaN,NaN,NaN,NaN
2,0001,00.01,Therapeutic ultrasound of vessels of head and neck,"[diagnostic ultrasound of: eye (95.13), diagnostic ultrasound of: head and neck (88.71), Therapeutic ultrasound of vessels of head and neck that of inner ear (20.79), ultrasonic: angioplasty of non-coronary vessel (39.50), ultrasonic: embolectomy (38.01, 38.02), ultrasonic: endarterectomy (38.11, 38.12), ultrasonic: thrombectomy (38.01, 38.02), diagnostic ultrasound (non-invasive) (88.71-88.79), intracardiac echocardiography [ICE] (heart chamber(s)) (37.28), intravascular imaging (adjunctive) (00.21-00.29)]",B,"[Anti-restenotic ultrasound, Intravascular non-ablative ultrasound]",NaN,NaN,NaN,NaN
3,0002,00.02,Therapeutic ultrasound of heart,"[diagnostic ultrasound of heart (88.72), ultrasonic ablation of heart lesion (37.34), ultrasonic angioplasty of coronary vessels (00.66, 36.09), diagnostic ultrasound (non-invasive) (88.71-88.79), intracardiac echocardiography [ICE] (heart chamber(s)) (37.28), intravascular imaging (adjunctive) (00.21-00.29)]",B,"[Anti-restenotic ultrasound, Intravascular non-ablative ultrasound]",NaN,NaN,NaN,NaN
4,0003,00.03,Therapeutic ultrasound of peripheral vascular vessels,"[diagnostic ultrasound of peripheral vascular system (88.77), ultrasonic angioplasty of: non-coronary vessel (39.50), diagnostic ultrasound (non-invasive) (88.71-88.79), intracardiac echocardiography [ICE] (heart chamber(s)) (37.28), intravascular imaging (adjunctive) (00.21-00.29)]",B,"[Anti-restenotic ultrasound, Intravascular non-ablative ultrasound]",NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
4641,9995,99.95,Stretching of foreskin,NaN,B,NaN,NaN,NaN,NaN,NaN
4642,9996,99.96,Collection of sperm for artificial insemination,NaN,M,NaN,NaN,NaN,NaN,NaN
4643,9997,99.97,Fitting of denture,NaN,B,NaN,NaN,NaN,NaN,NaN
4644,9998,99.98,Extraction of milk from lactating breast,NaN,F,NaN,NaN,NaN,NaN,NaN


### Pattern: Left + Right = Multiple

In [3]:
pattern = r'\bleft\b|\bright\b|\bmultiple\b'
filtered_icd9 = icd9[icd9['title'].str.contains(pattern, case=False, na=False)].copy()
filtered_icd9['base_group'] = filtered_icd9['Code'].astype(str).str[:-1]

def assign_type(title):
    title_lower = title.lower()
    if 'left' in title_lower: return 'left'
    if 'right' in title_lower: return 'right'
    if 'multiple' in title_lower: return 'combine'
    return None

filtered_icd9['type'] = filtered_icd9['title'].apply(assign_type)
filtered_icd9 = filtered_icd9.dropna(subset=['type'])

pivoted = filtered_icd9.pivot_table(
    index='base_group', 
    columns='type', 
    values=['Code', 'title'], 
    aggfunc='first' 
)

pivoted.columns = [f'{val.lower()}_{col}' for val, col in pivoted.columns]

# Ensure the columns generated by the pivot exist before dropping NaNs
required_cols = ['code_left', 'code_right', 'code_combine']
pivoted = pivoted.dropna(subset=required_cols)

# 5. Format final output table
final_table = pivoted.reset_index()
final_table = final_table[[
    'base_group', 
    'code_left', 'code_right', 'code_combine', 
    'title_left', 'title_right', 'title_combine'
]]

final_table

,base_group,code_left,code_right,code_combine,title_left,title_right,title_combine
0,173,1735,1733,1731,Laparoscopic left hemicolectomy,Laparoscopic right hemicolectomy,Laparoscopic multiple segmental resection of large intestine
1,457,4575,4573,4571,Open and other left hemicolectomy,Open and other right hemicolectomy,Open and other multiple segmental resection of large intestine


In [4]:
output_filename = '../result/combine_code_summary.xlsx'
final_table.to_excel(output_filename, index = False)

In [5]:
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

# 1. Load your existing Excel file
file_path = output_filename
wb = load_workbook(file_path)

print(f"Applying formatting to {file_path}...")

# 2. Loop through all the sheets in your workbook
for sheet_name in wb.sheetnames:
    ws = wb[sheet_name]

    # --- BEATIFICATION STEP A: Auto-fit Column Widths ---
    for col in ws.columns:
        max_length = 0
        column = col[0].column_letter # Gets the letter (A, B, C...)
        
        # Find the longest string of text in the entire column
        for cell in col:
            try:
                if len(str(cell.value)) > max_length:
                    max_length = len(str(cell.value))
            except:
                pass
        
        # Add a little padding (+2) and set the new width. 
        # (We cap it at 75 so extremely long inclusion texts don't stretch the screen too far)
        adjusted_width = min(max_length + 2, 75) 
        ws.column_dimensions[column].width = adjusted_width

    # --- BEATIFICATION STEP B: Style the Header Row ---
    # Create a light gray fill and a bold font
    header_fill = PatternFill(start_color="E0E0E0", end_color="E0E0E0", fill_type="solid")
    header_font = Font(bold=True, color="000000")

    # Apply the styling to every cell in the first row
    for cell in ws[1]: 
        cell.font = header_font
        cell.fill = header_fill
        # Center the text and wrap it if it gets too long
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

# 3. Save the changes back to the file
wb.save(file_path)
print("Excel file successfully beautified!")

Applying formatting to ../result/combine_code_summary.xlsx...
Excel file successfully beautified!


In [7]:
final_table
{row['code_combine'] : [row['code_left'], row['code_right']] for i,row in final_table.iterrows()}

{'1731': ['1735', '1733'], '4571': ['4575', '4573']}